# Mapping the Potential Destructive Power of Wildfires Using Machine Learning

## Appendix: *Gridmet Extraction*
- *Version Number: 3.0*
---
### Contents  
> 1. *Download nc files* 
> 2. *Extract weather variables*
> 3. *Drought Indexes Extracted from manually downloaded nc files*
---
### Notes
---
### Inputs
> - **gridMET**: <https://www.climatologylab.org/gridmet.html
---
### Outputs  
- `Environmental.csv` - cleaned dataset of environmental variables 
---
### User Created Dependencies  
---

### Third Party Dependencies  
---

In [1]:
import os
import pandas as pd
import geopandas as gpd
import xarray as xr

## Data Processing

*Located in:* 
> - *notebooks/01_Fire_Damage_Processing.ipynb*
> - *notebooks/02_Weather_Data_Processing.ipynb*
> - *notebooks/A_Appendix.pynb*

- ArcGIS, constructed an equally spaced grid of sampling points for overall better coverage.
- Merged detailed fire records with sampling points via intersect spatial join.
- Imputed missing values for weather stations.

In [1]:
import xarray as xr

ds = xr.open_dataset("gridmet_raw/tmmx_2018.nc", engine="netcdf4")
print(ds)


<xarray.Dataset>
Dimensions:          (lon: 1386, lat: 585, day: 365, crs: 1)
Coordinates:
  * lon              (lon) float64 -124.8 -124.7 -124.7 ... -67.14 -67.1 -67.06
  * lat              (lat) float64 49.4 49.36 49.32 49.28 ... 25.15 25.11 25.07
  * day              (day) datetime64[ns] 2018-01-01 2018-01-02 ... 2018-12-31
  * crs              (crs) uint16 3
Data variables:
    air_temperature  (day, lat, lon) float32 ...
Attributes: (12/19)
    geospatial_bounds_crs:      EPSG:4326
    Conventions:                CF-1.6
    geospatial_bounds:          POLYGON((-124.7666666333333 49.40000000000000...
    geospatial_lat_min:         25.066666666666666
    geospatial_lat_max:         49.40000000000000
    geospatial_lon_min:         -124.7666666333333
    ...                         ...
    date:                       25 April 2021
    note1:                      The projection information for this file is: ...
    note2:                      Citation: Abatzoglou, J.T., 2013, Develo

## Download and Extract

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import xarray as xr
import requests
from tqdm import tqdm

GRIDMET_VARIABLES = [
    "tmmx", "tmmn", "vpd", "rmax", "rmin", "sph",
    "pr","vs","srad", "etr",
    "bi", "erc", "fm100", "fm1000"]

DOWNLOAD_DIR = "gridmet_raw"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

POINTS_FILE = "../data/raw/points.gpkg"
OUTPUT_CSV = "../data/raw/gridmet_point_extraction.csv"

# LOAD POINTS

gdf = gpd.read_file(POINTS_FILE)

# Ensure required columns exist
assert "Sample_ID" in gdf.columns, "Missing Sample_ID column"
assert "Date" in gdf.columns, "Missing Date column"
assert "geometry" in gdf.columns, "Missing geometry column"

# Extract lat/lon
gdf["lon"] = gdf.geometry.x
gdf["lat"] = gdf.geometry.y

# Convert Date to datetime
gdf["Date"] = pd.to_datetime(gdf["Date"])

# List years needed
years = sorted(gdf["Date"].dt.year.unique())

# FUNCTION: DOWNLOAD A SINGLE GRIDMET FILE

def download_gridmet_file(var, year):
    filename = f"{var}_{year}.nc"

    url = f"http://www.northwestknowledge.net/metdata/data/{var}_{year}.nc"
    out_path = os.path.join(DOWNLOAD_DIR, filename)

    if os.path.exists(out_path):
        return out_path

    print(f"[Downloading] {filename}")

    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()

        total_size = int(response.headers.get("content-length", 0))
        with open(out_path, "wb") as f, tqdm(
            total=total_size, unit="B", unit_scale=True, desc=filename
        ) as pbar:
            for chunk in response.iter_content(1024):
                f.write(chunk)
                pbar.update(len(chunk))

    except Exception as e:
        print(f"ERROR downloading {filename}: {e}")
        if os.path.exists(out_path):
            os.remove(out_path)

    return out_path

# EXTRACT VALUES

records = []

for var in GRIDMET_VARIABLES:
    print(f"\n=== Extracting variable: {var} ===")

    # Download needed years
    for yr in years:
        download_gridmet_file(var, yr)

    # Process each year independently
    for yr in years:
        filepath = os.path.join(DOWNLOAD_DIR, f"{var}_{yr}.nc")
        ds = xr.open_dataset(filepath)

        subset = gdf[gdf["Date"].dt.year == yr]

        for idx, row in subset.iterrows():
            date = row["Date"]
            lat = row["lat"]
            lon = row["lon"]
            sid = row["Sample_ID"]

            try:
                value = ds[var].sel(
                    time=date,
                    lat=lat,
                    lon=lon,
                    method="nearest"
                ).values.item()
            except Exception:
                value = None

            records.append({
                "Sample_ID": sid,
                "Date": date,
                "Variable": var,
                "Value": value
            })


df = pd.DataFrame(records)

wide = df.pivot_table(
    index=["Sample_ID", "Date"],
    columns="Variable",
    values="Value"
).reset_index()

wide.to_csv(OUTPUT_CSV, index=False)

print(f"\Extraction complete!")
print(f"CSV saved to: {OUTPUT_CSV}")


## Extract Only

In [ ]:
import os
import pandas as pd
import geopandas as gpd
import xarray as xr
from tqdm import tqdm

GRIDMET_FOLDER = r"C:\Users\dusti\Desktop\Cal_Fire\notebooks\gridmet_raw"

# Mapping from file prefix to variable name inside NetCDF
VARIABLE_LOOKUP = {
    "bi": "burning_index_g",
    "erc": "energy_release_component-g",
    "etr": "potential_evapotranspiration",
    "fm1000": "dead_fuel_moisture_1000hr",
    "fm100": "dead_fuel_moisture_100hr",
    "pr": "precipitation_amount",
    "rmax": "relative_humidity",
    "rmin": "relative_humidity",
    "sph": "specific_humidity",
    "srad": "surface_downwelling_shortwave_flux_in_air",
    "tmmn": "air_temperature",
    "tmmx": "air_temperature",
    "vpd": "mean_vapor_pressure_deficit",
    "vs": "wind_speed"
}

POINTS_FILE = "../data/raw/points.gpkg"  # CHANGE IF NEEDED
OUTPUT_CSV = "../data/raw/gridmet_point_extraction.csv"

# LOAD POINTS

gdf = gpd.read_file(POINTS_FILE)

# Ensure required columns exist
assert "Sample_ID" in gdf.columns, "Missing Sample_ID column"
assert "Date" in gdf.columns, "Missing Date column"
assert "geometry" in gdf.columns, "Missing geometry column"

# Extract lat/lon
gdf["lon"] = gdf.geometry.x
gdf["lat"] = gdf.geometry.y

# Convert Date to datetime
gdf["Date"] = pd.to_datetime(gdf["Date"])

years = sorted(gdf["Date"].dt.year.unique())

# EXTRACT VALUES

records = []

for prefix, var_name in VARIABLE_LOOKUP.items():
    print(f"\n=== Extracting variable: {prefix} ({var_name}) ===")


    for yr in years:
        filepath = os.path.join(GRIDMET_FOLDER, f"{prefix}_{yr}.nc")
        if not os.path.exists(filepath):
            print(f"WARNING: {filepath} not found, skipping")
            continue

        ds = xr.open_dataset(filepath)

        subset = gdf[gdf["Date"].dt.year == yr]

        for idx, row in tqdm(subset.iterrows(), total=len(subset)):
            date = row["Date"]
            lat = row["lat"]
            lon = row["lon"]
            sid = row["Sample_ID"]

            try:
                value = ds[var_name].sel(
                    day=date,  
                    lat=lat,
                    lon=lon,
                    method="nearest"
                ).values.item()
            except Exception:
                value = None

            records.append({
                "Sample_ID": sid,
                "Date": date,
                "Variable": prefix,
                "Value": value
            })

# SAVE LONG → WIDE CSV

df = pd.DataFrame(records)

wide = df.pivot_table(
    index=["Sample_ID", "Date"],
    columns="Variable",
    values="Value"
).reset_index()

wide.to_csv(OUTPUT_CSV, index=False)

print(f"\Extraction complete! CSV saved to: {OUTPUT_CSV}")


## Drought Indexes Processed Separately
downloaded manually due to different file structure

In [ ]:
GRIDMET_FOLDER = r"C:\Users\dusti\Desktop\Cal_Fire\notebooks\gridmet_raw"
['spi30d','spi180d','spei30d','spei90d','spei180d''pdsi']

In [ ]:
GRIDMET_FOLDER = r"C:\Users\dusti\Desktop\Cal_Fire\notebooks\gridmet_raw"

# Your drought files
DROUGHT_FILES = [
    "spi30d.nc",
    "spi180d.nc",
    "spei30d.nc",
    "spei90d.nc",
    "spei180d.nc",
    "pdsi.nc"
]

variables_dict = {}

for fname in DROUGHT_FILES:
    full_path = os.path.join(GRIDMET_FOLDER, fname)
    print(f"\nOpening: {fname}")
    
    try:
        ds = xr.open_dataset(full_path)
        # List variable names excluding coordinates/aux variables
        var_list = list(ds.data_vars.keys())
        
        variables_dict[fname] = var_list
        
        print(f"Variables found: {var_list}")
    except Exception as e:
        print(f"Error opening {fname}: {e}")

print("\n\n=== COMPLETE VARIABLE LIST ===")
for f, vars in variables_dict.items():
    print(f"{f}: {vars}")


In [ ]:
DROUGHT_DATASETS = {
    "spi30d.nc":   ("spi30d", "spi30d_category"),
    "spi180d.nc":  ("spi180d", "spi180d_category"),
    "spei30d.nc":  ("spei30d", "spei30d_category"),
    "spei90d.nc":  ("spei90d", "spei90d_category"),
    "spei180d.nc": ("spei180d", "spei180d_category"),
    "pdsi.nc":     ("pdsi", "pdsi_category")
}


In [ ]:
BASE_FOLDER = r"C:\Users\dusti\Desktop\Cal_Fire\notebooks\gridmet_raw"

FNAME = "pdsi.nc"
VAR_NAME = "pdsi"  # The main variable inside the file

POINTS_FILE = "../data/raw/points.gpkg"
OUTPUT_CSV = "../data/raw/pdsi_extraction.csv"


gdf = gpd.read_file(POINTS_FILE)
gdf["Date"] = pd.to_datetime(gdf["Date"])
gdf["lon"] = gdf.geometry.x
gdf["lat"] = gdf.geometry.y

records = []

ds_path = os.path.join(BASE_FOLDER, FNAME)
ds = xr.open_dataset(ds_path)

# Find time dimension (usually named 'day')
time_dim = [t for t in ds.coords if ds.coords[t].dtype.kind == "M"][0]


for idx, row in gdf.iterrows():
    date = row["Date"]
    lat = row["lat"]
    lon = row["lon"]
    sid = row["Sample_ID"]

    try:
        val = ds[VAR_NAME].sel(
            {time_dim: date, "lat": lat, "lon": lon},
            method="nearest"
        ).values.item()
    except:
        val = None

    records.append({
        "Sample_ID": sid,
        "Date": date,
        VAR_NAME: val
    })


df = pd.DataFrame(records)
df.to_csv(OUTPUT_CSV, index=False)

print("\ DONE — extracted single variable!")
print(f"Saved to: {OUTPUT_CSV}")